# A Laser Wakefield Accelerator

This is a small **3D teaching example** of laser wakefield acceleration. 
A short laser pulse interacts with a plasma. 
The laser pushes the plasma electrons aside and they oscillate behind it, creating a wake that can accelerate electrons. 
A short density drop changes the wake and can enable trapping.

We will look at electron density, the accelerating field, and the energy spectrum.

Run the simulation in a terminal, then use this notebook from
`htu/lwfa_warpx/` to analyze the saved output. Select a kernel
with NumPy, Matplotlib, SciPy, and `openpmd-viewer`; plotting does not need a GPU.


In [ ]:
import matplotlib.pyplot as plt
from openpmd_viewer import OpenPMDTimeSeries
from scipy.constants import e

## Run the simulation in a terminal

From `htu/lwfa_warpx/`, activate your WarpX environment and run:

```bash
time python run_lwfa_warpx.py > run.log 2>&1
```

The driver reads `lwfa_warpx_input.txt` and writes to `diags/`. It does not
create a fresh directory on each run. Before a repeat, preserve the completed
run's `diags` folder and `run.log` under new, unused names. Record the elapsed
(`real`) time, inspect the log for successful completion, and measure output
size with `du -sh diags`.

For an NVIDIA GPU run, use `nvidia-smi` in a second terminal to inspect GPU
activity. You can also run with a native `warpx.3d lwfa_warpx_input.txt` executable. Once the run finishes, execute the
analysis cells below. The next cell locates `htu/lwfa_warpx/` and sets an absolute diagnostics path; it also works when the kernel starts in `htu/`.


In [ ]:
import sys
from pathlib import Path

# Locate this notebook's folder even if the kernel started in htu/.
notebook_dir = Path.cwd()
if not (notebook_dir / "warpx_helpers.py").is_file():
    notebook_dir = notebook_dir / "lwfa_warpx"
sys.path.insert(0, str(notebook_dir))
out_folder = str(notebook_dir / "diags/diag1")
if not Path(out_folder).is_dir():
    raise FileNotFoundError(f"WarpX diagnostics not found at {out_folder}. Run the simulation first or update out_folder.")
print("Reading diagnostics from:", out_folder)


## Open the diagnostics and plot particles

Start with the saved data directly. [OpenPMDTimeSeries](https://openpmd-viewer.readthedocs.io/en/latest/tutorials/1_Introduction-to-the-API.html) lists the available iterations and species. `get_particle(...)` returns one array per requested quantity, with one entry per macroparticle. Positions are in meters; `w` is the number of physical electrons represented by each macroparticle.

Choose a saved step, then extract the longitudinal and transverse positions. This includes **all saved electrons**, including plasma electrons; it is not a selection of a trapped bunch.


In [ ]:
series = OpenPMDTimeSeries(out_folder)
print("Saved steps:", series.iterations)
print("Species:", series.avail_species)
iteration = int(series.iterations[len(series.iterations) // 2])
z, x, w = series.get_particle(["z", "x", "w"], species="electrons", iteration=iteration)
print(f"Read {len(x):,} macroparticles at step {iteration}")


Plot a two-dimensional histogram with Matplotlib. Convert meters to micrometers and particle weights to charge magnitude in pC. Each pixel shows **charge per bin**, integrated over the unplotted y direction.

The horizontal bands reveal the regular macroparticle loading: this input uses one particle per cell and only 32 cells across x, while the histogram uses 80 x-bins. Many background electrons remain close to their initial rows. These bands are not separate physical electron beams. The density-field plots below use the deposited charge density instead of binning raw particle positions.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
hist = ax.hist2d(z * 1e6, x * 1e6, bins=80, weights=w * e * 1e12, cmap="magma")
fig.colorbar(hist[3], ax=ax, label="Charge per bin (pC)")
ax.set(xlabel="z (µm)", ylabel="x (µm)", title=f"Saved electrons · step {iteration}")
plt.show()


## Read and plot the laser field

The diagnostics also save the electric field `E`. The laser is polarized along y, so read `E_y` at the same step and take a slice through y = 0. `get_field(...)` returns both the field array (V/m) and its coordinate metadata. This is the total transverse electric field, including any plasma contribution, rather than a separately extracted laser envelope.


In [ ]:
laser, info = series.get_field("E", coord="y", iteration=iteration, slice_across="y")
# imshow needs x along the rows and z along the columns.
if info.axes[0] == "z":
    laser = laser.T

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
scale = max(abs(laser).max() / 1e9, 1e-12)
im = ax.imshow(
    laser / 1e9, origin="lower", aspect="auto", cmap="RdBu_r",
    extent=[info.z[0] * 1e6, info.z[-1] * 1e6, info.x[0] * 1e6, info.x[-1] * 1e6],
    vmin=-scale, vmax=scale,
)
fig.colorbar(im, ax=ax, label="Electric field Ey (GV/m)")
ax.set(xlabel="z (µm)", ylabel="x (µm)", title=f"Laser field, y = 0 · step {iteration}")
plt.show()


## Watch the wake evolve

These three density slices share one color scale. The horizontal coordinate
`z-ct` follows the moving window. Look for the low-density cavity and the
concentrated electrons behind it.

In [ ]:
from warpx_helpers import plot_density_evolution, plot_snapshot

fig = plot_density_evolution(out_folder)
plt.show()

## Look at the wake

Change `iteration` to any saved step and rerun the cell. The middle frame
usually shows the wake inside the plasma; the final frame follows the beam
into vacuum. White contours show |Ey|, including both laser and plasma fields. The spectrum includes
all electrons still in the moving box, including untrapped plasma electrons.

In [ ]:
iteration = int(series.iterations[len(series.iterations) // 2])
fig = plot_snapshot(out_folder, iteration)
plt.show()